# Workflow of this pipeline

1) **resave** `nd2` images as `tif`, splitting individual channels

2) create RS-FISH detection settings `Log.txt` file in Fiji (independent of this pipeline)

-> **detect spots** using RS-FISH producing:
- `/.../detections` folder containing a `csv` with spot information for each provided `tif`
- `merge.csv` combining all `csvs` in the `detections` folder

3) **visualise the detections**, creating a `/.../detections/vis` folder containing `png` max projections of images in the spot file (eg. `merge.csv`)

4) **segment nuclei** with cellpose, using `tif`s and a cellpose classifyer (default or costum trained)
5) **filter spots not in nuclei** and calculate sensitivity based on spots from (3) and segmentation masks from (4)
6) spots in 2 channels are matched based on the closest neighbour and **spot distances calculated**
7) **remove tif folder** to save space

In [ ]:
import json
import os
from glob import glob
import sys
import tifffile
import pandas as pd
import numpy as np
import torch

sys.path.append('/home/stumberger/fish-pipelines/')
from fish_utils.resave import resave_nd2, remove_tifs
from fish_utils.spot_detection import read_parameters, make_fiji_command, detect_spots, combine_csv, create_folder, plot_detections
from fish_utils.spot_analysis import add_cell_info, get_sensitivity, detect_spot_pairs

from natsort import natsorted
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from skimage.io import imread

from cellpose import models, io
from cellpose.io import imread
from cellpose import plot

# 1) Split nd2 channels and resave as tif

In [ ]:
# specify the upper level folder, containing the "raw" folder with the images
path = "/scratch/stumberger/spinning_disk_spot_detection/v2/"

resave_nd2(path)

# 2) Spot detection
Before running this, open one of your images in fiji, run RS-FISH on it, and save the `Log.txt` file with the spot detection parameters for each of the spot channels- **IMPORTAINT!**

In [ ]:
path = "/scratch/stumberger/spinning_disk_spot_detection/v2/" #upper level experiment folder

detection_settings = ["/scratch/stumberger/spinning_disk_spot_detection/v2/Log_ch2.txt",
                     "/scratch/stumberger/spinning_disk_spot_detection/v2/Log_ch2.txt"]
channels = [1,2] # which channel to detect spots in (counting starts at 0!)

# run
detect_spots(path,detection_settings,channels,
            macro_path = "/home/stumberger/fish-pipelines/fish_utils/RS_macro_param.ijm",
            fiji_path = "/home/stumberger/tools/Fiji.app/ImageJ-linux64")

# combine all csvs in a folder into 1
combine_csv(path)

# 3) Visualise detections
Create visualisation to check how well the spot detection worked. 

In [ ]:
path = "/scratch/stumberger/spinning_disk_spot_detection/v2/" #upper level experiment folder
path_spots= None #path to `merge.csv` file; if None defaults to /.../detections/merge.csv 
out_folder = None #where to save visualisations; if None defaults to /.../detections/vis
channels = [2] # which channel to plot spots for (counting starts at 0!)
range_quantiles = (0.02, 0.9999) #if the spots are not well visible you can rescale image intensity

plot_detections(path,channels)

# 4) Segmentation
Segmentation needed for further spot filtering and calculations.

In [ ]:
# to do the segmentaion fast work on the gpu
device = torch.device('cuda:1')
torch.cuda.is_available()

In [ ]:
model = models.Cellpose(model_type='nuclei', device=device)

files = glob('/scratch/stumberger/spinning_disk_spot_detection/v2/tif/*_ch0.tif') #DAPI tif
out_folder = "/scratch/stumberger/spinning_disk_spot_detection/v2/segmentation/" #where to save visualisation
create_folder(out_folder)

channels = [[0,0]]

# or in a loop
for filename in files:
    
    out = filename.replace("/tif", "/segmentation")
    img = io.imread(filename).max(axis=0)
    
    masks, flows, styles, diams = model.eval(img, diameter=200, channels=channels,flow_threshold=0.5)

    # save results so you can load in gui
    io.masks_flows_to_seg(img, masks, flows, diams, out, channels)

    # save results as png
    io.save_to_png(img, masks, flows, out)
    
    # plot segmentation o check
    fig = plt.figure(figsize=(12,3.5))
    plot.show_segmentation(fig, img, masks, flows[0], channels=channels)
    plt.tight_layout()
    fig.savefig(f"{out_folder}/vis/{os.path.basename(out)}.png",dpi=300)
    plt.close(fig)

# 5) Filter spots based on segmentation
Here you can exclude spots outside of nuclei based on the segmentation and calculate the spot sensitivity. 
You can repeat step 3 after this to visualise only spots inside nuclei. 

In [ ]:
## add cell info to spots ##
path = "/scratch/stumberger/spinning_disk_spot_detection/v2/" #upper level experiment folder
path_spots = f"{path}/detections/merge.csv" #spots file
masks = glob(f"{path}/segmentation/*.npy") #all segmentation masks (.npy and .png supported)
out = f"{path}/detections/spots_filtered.csv" # where to save spots

#filter=True - exclude spots outside of nuclei
add_cell_info(masks,path_spots,out,filter=True,mask_ending="_seg")

In [ ]:
## calculate sensitivity ##
path_spots = f"{path}/detections/spots_filtered.csv"
out = f"{path}/detections/spots_sensitvity.csv"
get_sensitivity(path_spots, out)

# 6) Calculate spot distances
Spots in 2 channels are matched based on the closest neighbour and distances calculated.

In [ ]:
path = "/scratch/stumberger/spinning_disk_spot_detection/v2/" #upper level experiment folder
path_spots = f"{path}/detections/merge.csv"
out_distances = f"{path}/distances.csv"
channels = [1,2] # which channels to match
voxel_size=(300, 130, 130) #sizes of zyx [nm]

detect_spot_pairs(path_spots,out_distances,channels,voxel_size)

# 7) Remove tif folder 
Please always remove tifs afetr you are done with the analysis to save space.

In [ ]:
folder = "/data/agl_data/../tif/"

remove_tifs(folder)